# abl_duoi_run — CHẠY NỐT 2 MODEL CÒN THIẾU (`jina-v2`, `gte-multi`) · **BẢN VÁ 2**

Lượt trước cả hai đều hỏng trong 256 giây. **Không cái nào hỏng vì phần cứng** —
đổi T4/P100 không cứu được. Log chỉ đúng hai lỗi phần mềm, đã vá cả hai:

| Model | Lỗi thật | Miếng vá |
|---|---|---|
| `jina-v2` | `ImportError: create_position_ids_from_input_ids` — mã tuỳ biến gọi hàm private mà `transformers` mới đã bỏ. Hỏng lúc **import**, chưa chạm GPU. | Cấy lại hàm đó (3 dòng, nguyên bản cũ) vào module trước khi tải |
| `gte-multi` | `device-side assert: index out of bounds`. Config `type_vocab_size = 1` nhưng tokenizer chấm **cặp** nên sinh `token_type_ids` = 0 **và 1** → tra bảng cỡ 1 bằng chỉ số 1 → ngoài biên | Bỏ hẳn `token_type_ids` khỏi đầu vào khi `type_vocab_size <= 1` |

> `gte-multi` là **đúng họ lỗi đã giết PhoRanker** hôm 23/08 — tra bảng nhúng ngoài biên.
> Khác mỗi bảng: lần đó bảng **vị trí**, lần này bảng **loại token**. `gioi_han()` chỉ canh
> bảng vị trí nên không bắt được. Đã ghi vào CLAUDE.md như một họ lỗi, không phải ca lẻ.

### Cách chạy

1. Dataset giữ NGUYÊN (`project-ir`). Không cần upload gì.
2. Accelerator: `GPU T4 ×2` hay `P100` đều được — **không liên quan tới hai lỗi này**.
3. **ENVIRONMENT giữ nguyên `Pin to original environment`.** Miếng vá 1 được viết cho
   đúng bản `transformers` đó; bỏ pin là vá sai chỗ.
4. Save & Run All. Xong tải `outputs/` gửi lại.

### Nếu vẫn hỏng

**Dừng hẳn, bỏ hai dòng này.** Bảng 9/11 model đã đủ cho báo cáo. Đây là lần thử thứ hai
và là lần cuối — đừng đốt thêm giờ GPU vì hai dòng tô điểm.

Chi phí: dựng 15.000 cặp ~3 phút + 2 model ~5 phút. **Dưới 15 phút.**


In [ ]:
!pip install -q -U sentence-transformers underthesea

In [ ]:
# ===== Bước 1: dựng 15.000 cặp — DÙNG CHUNG cho mọi model =====
import os, sys, json, time, gc, glob, traceback
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"thiết bị: {DEV}" + (f" · {torch.cuda.get_device_name(0)}" if DEV == "cuda" else ""))

# KHÁC oracle13: lượt này 15.000 cặp × 9 model. Trên CPU là ~3,5 GIỜ MỖI MODEL = hơn 30h.
# Không chặn thì nó cứ thế chạy cả đêm rồi hết phiên mà chưa xong model thứ hai.
CHO_PHEP_CPU = False
assert DEV == "cuda" or CHO_PHEP_CPU, (
    "CẦN GPU. 15.000 cặp/model trên CPU ~3,5h mỗi model. "
    "Settings -> Accelerator -> GPU. (Cố tình chạy CPU thì đặt CHO_PHEP_CPU = True.)")

EXC = 900                                   # ký tự trích mỗi văn bản, y hệt hard15/enrich
INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

import deep_chunk as DC
from rerank import load_reranker
from rerank_from_d import blend_bm25_first
DC.MERGE_CHARS = 1800

dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
cand = json.load(open(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000.json", encoding="utf-8"))
qids = [q for q in dev if q in cand]
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in qids}
ORDER= {q: [str(c["doc_id"]) for c in
            sorted(cand[q], key=lambda c: -float(c["rrf_score"]))] for q in qids}

t0 = time.time()
index, texts = [], []
for i, q in enumerate(qids, 1):
    qs = dev[q]["question"]
    for d in ORDER[q]:
        e = DC.pick_chunks(qs, CTX_DIR, d, k=1)
        index.append((q, d)); texts.append((e[0] if e else "")[:EXC])
    if i % 100 == 0: print(f"  băm {i}/{len(qids)} | {(time.time()-t0)/60:.1f} phút", flush=True)
QTEXT = [dev[q]["question"] for q, _ in index]
print(f"\n{len(qids)} câu × 50 = {len(index):,} cặp/model")
print(f"TRẦN rổ 50: {sum(len(GOLD[q] & set(ORDER[q]))/len(GOLD[q]) for q in qids)/len(qids):.4f}")

In [ ]:
# ===== Bước 2: chỉ số + tách từ (cho họ PhoBERT) =====
def do(per, n):
    """recall@1/5/10 + MRR@10 cho một bảng điểm {qid: {doc: score}}."""
    r = {1: 0.0, 5: 0.0, 10: 0.0}; mrr = 0.0
    for q in qids:
        rk = sorted(per[q], key=lambda d: -per[q][d])
        p10 = blend_bm25_first(rk, ORDER[q], k=10, n_bm25=n)
        for k in r: r[k] += len(GOLD[q] & set(p10[:k])) / len(GOLD[q])
        hit = next((i for i, d in enumerate(p10, 1) if d in GOLD[q]), 0)
        mrr += 1.0 / hit if hit else 0.0
    m = len(qids)
    return {"R@1": r[1]/m, "R@5": r[5]/m, "R@10": r[10]/m, "MRR@10": mrr/m}

def gioi_han(mid, want, trc=False):
    """Hạ max_length xuống trần THẬT của kiến trúc, đọc từ config trước khi tải trọng số.

    Đây là chỗ đã giết `PhoRanker` (PhoBERT `max_position_embeddings`=258 mà bị đưa 512):
    quá trần thì tra bảng vị trí ngoài biên -> device-side assert, hỏng CUDA context.
    Bảng tra tay thì sớm muộn cũng sót một model; đọc thẳng từ config thì không.
    `ms-marco-MiniLM` (BERT tiếng Anh, trần 512) đặc biệt nguy: tokenizer tiếng Anh băm
    tiếng Việt có dấu ra RẤT nhiều mảnh, 900 ký tự có thể vọt quá 512 token.
    """
    try:
        from transformers import AutoConfig
        lim = getattr(AutoConfig.from_pretrained(mid, trust_remote_code=trc),
                      "max_position_embeddings", None)
        if lim and lim < 10_000:
            want = min(want, lim - 2)
    except Exception as e:
        print(f"    (không đọc được config {mid}: {type(e).__name__}) — giữ {want}")
    return want

def don_cache():
    """11 model ~15GB. Kaggle chỉ có ~20GB đĩa ghi -> dọn sau mỗi model, đừng để hết chỗ."""
    import shutil
    for p in ("/root/.cache/huggingface/hub", "/root/.cache/torch/sentence_transformers"):
        shutil.rmtree(p, ignore_errors=True)

try:                                        # PhoBERT cần văn bản ĐÃ TÁCH TỪ, không thì điểm rác
    from functools import lru_cache
    from underthesea import word_tokenize
    # nhớ kết quả: 300 câu hỏi nhưng bị lặp 50 lần mỗi câu -> không cache là tách thừa 299/300
    seg = lru_cache(maxsize=None)(lambda s: word_tokenize(s, format="text"))
    print("underthesea OK — chạy được nhánh PhoBERT")
except Exception as e:
    seg = None
    print(f"KHÔNG có underthesea ({e}) — bỏ qua PhoRanker/ViRanker")

In [ ]:
# ===== Bước 3: chạy từng model. Hỏng một cái KHÔNG giết cả bảng =====
# (tag, model_id, kwargs, cần_tách_từ)  — XẾP THEO CHI PHÍ TĂNG DẦN, hết giờ thì cắt đuôi
# Kiến trúc CHUẨN trước (an toàn), `trust_remote_code` XUỐNG CUỐI: 23/08 `gte-multi` bắn
# device-side assert làm hỏng luôn CUDA context, mọi model sau nó đều chết theo.
MODELS = [
 ("jina-v2",   "jinaai/jina-reranker-v2-base-multilingual",
                dict(trust_remote_code=True, max_length=512), False),
 ("gte-multi", "Alibaba-NLP/gte-multilingual-reranker-base",
                dict(trust_remote_code=True, max_length=512), False),
]
# LƯỢT ĐUÔI 24/08 — 9 model kia XONG rồi, kết quả đã tải về máy. Lượt này CHỈ chạy 2 cái
# còn thiếu nên KHÔNG cần upload lại outputs/ cũ lên dataset: không có gì để bỏ qua.
# max_length ÉP 512: trích 900 ký tự ~ 300-450 token nên 512 không hề cắt mất gì,
# đổi lại loại sạch khả năng tra bảng vị trí ngoài biên — nghi phạm số 1 của device-side
# assert đã giết gte-multi hôm 23/08. jina đặt TRƯỚC vì gte mới là cái từng làm sập.

# ================== HAI MIẾNG VÁ CHO ĐÚNG HAI LỖI CỦA LƯỢT 24/08 ==================
# Cả hai model đều KHÔNG phải hỏng vì phần cứng. Log nói rất rõ:
#
# jina-v2   : ImportError `create_position_ids_from_input_ids` — mã tuỳ biến của jina
#             gọi một hàm PRIVATE của transformers mà bản mới đã bỏ. Lỗi phiên bản,
#             xảy ra lúc import, chưa hề chạm GPU.
# gte-multi : device-side assert "index out of bounds" ngay forward đầu tiên.
#             Nguyên nhân: config có `type_vocab_size = 1` (chỉ một loại token), nhưng
#             tokenizer chấm CẶP (câu hỏi, văn bản) nên sinh token_type_ids = 0 VÀ 1.
#             Tra bảng nhúng cỡ 1 bằng chỉ số 1 -> ngoài biên -> assert. Đúng họ lỗi
#             đã giết PhoRanker, chỉ khác bảng: lần đó là bảng VỊ TRÍ, lần này là bảng LOẠI TOKEN.

import transformers.models.xlm_roberta.modeling_xlm_roberta as _xlmr
if not hasattr(_xlmr, "create_position_ids_from_input_ids"):
    def create_position_ids_from_input_ids(input_ids, padding_idx, past_key_values_length=0):
        # nguyên bản của transformers cũ, 3 dòng, chép lại y hệt
        mask = input_ids.ne(padding_idx).int()
        idx = (torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask
        return idx.long() + padding_idx
    _xlmr.create_position_ids_from_input_ids = create_position_ids_from_input_ids
    print("đã vá: create_position_ids_from_input_ids (cho jina-v2)")

def bo_token_type(m, tag):
    """type_vocab_size == 1 thì token_type_ids BẮT BUỘC toàn 0. Tokenizer chấm cặp lại
    sinh số 1 -> ngoài biên. Bỏ hẳn khoá đó khỏi đầu vào là xong, không đổi ngữ nghĩa."""
    tk = getattr(m, "tokenizer", None)
    cf = getattr(getattr(m, "model", None), "config", None)
    if tk is None or cf is None:
        return
    if getattr(cf, "type_vocab_size", 2) <= 1 and "token_type_ids" in getattr(tk, "model_input_names", []):
        tk.model_input_names = [x for x in tk.model_input_names if x != "token_type_ids"]
        print(f"    [{tag}] type_vocab_size={cf.type_vocab_size} -> đã bỏ token_type_ids")
# ==================================================================================

def cuda_con_song():
    """Một phép tính bé xíu. Sau device-side assert thì nó ném lỗi -> biết context đã chết."""
    if DEV != "cuda":
        return True
    try:
        torch.zeros(1, device="cuda").add_(1); torch.cuda.synchronize(); return True
    except Exception:
        return False

T0 = time.time()
RES = {}
for tag, mid, kw, need_seg in MODELS:
    if time.time() - T0 > 10 * 3600:        # trần 12h/lượt — dừng trước khi bị cắt ngang
        print("\n!! Đã chạy 10 giờ, dừng để kịp lưu. Upload outputs/ rồi chạy tiếp."); break
    p = f"{OUT}/scores_ablation_{tag}.json"
    # Tìm ĐỆ QUY trong dataset: upload cả thư mục `outputs/` thì file nằm ở
    # {INPUT_DIR}/outputs/..., tìm phẳng sẽ không thấy và chấm lại từ đầu.
    old = glob.glob(f"{INPUT_DIR}/**/scores_ablation_{tag}.json", recursive=True)
    src = p if os.path.isfile(p) else (old[0] if old else None)
    if src:
        mp = os.path.join(os.path.dirname(src), f"meta_ablation_{tag}.json")
        mt = json.load(open(mp, encoding="utf-8")) if os.path.isfile(mp) else {}
        raw = json.load(open(src, encoding="utf-8"))
        # nhận cả hai dạng: {doc: điểm} và {doc: {"ce":..,"bm25":..}} của enrich_run
        RES[tag] = ({q: {d: (v["ce"] if isinstance(v, dict) else v) for d, v in e.items()}
                     for q, e in raw.items()}, mid, mt)
        print(f"{tag}: đã có, bỏ qua"); continue
    if need_seg and seg is None:
        print(f"{tag}: BỎ — cần tách từ mà không có underthesea"); continue
    try:
        t0 = time.time()
        kw["max_length"] = gioi_han(mid, kw.get("max_length", 1024),
                                    kw.get("trust_remote_code", False))
        print(f"[{tag}] max_length = {kw['max_length']}", flush=True)
        m = load_reranker(mid, device=DEV, **kw)
        bo_token_type(m, tag)
        pr = [[seg(q), seg(t)] for q, t in zip(QTEXT, texts)] if need_seg else \
             [[q, t] for q, t in zip(QTEXT, texts)]
        sc = m.predict(pr)
        el = time.time() - t0
        per = {}
        for (q, d), v in zip(index, sc): per.setdefault(q, {})[d] = float(v)
        # đếm tham số TẠI CHỖ, đừng chép tay từ model card — đã có tiền lệ tên "4B" mà thật 4,02B
        mt = {"params": sum(x.numel() for x in m.model.parameters()),
              "sec_per_pair": el / len(index)}
        json.dump(per, open(p, "w", encoding="utf-8"), ensure_ascii=False)
        json.dump(mt, open(f"{OUT}/meta_ablation_{tag}.json", "w", encoding="utf-8"))
        RES[tag] = (per, mid, mt)
        print(f">>> {tag}: {el/60:.1f} phút · {mt['sec_per_pair']:.3f} s/cặp · "
              f"{mt['params']/1e9:.3f}B tham số -> đã lưu"
              f"   [tổng {(time.time()-T0)/60:.0f} phút]", flush=True)
    except Exception:
        print(f">>> {tag}: HỎNG\n{traceback.format_exc()[-500:]}")
    finally:
        # BẮT BUỘC bọc try: sau device-side assert thì CHÍNH empty_cache() cũng ném lỗi, mà
        # nó nằm NGOÀI except ở trên -> văng khỏi vòng lặp, giết cả bảng. Đã dính 23/08.
        globals().pop("m", None); globals().pop("pr", None)
        don_cache()
        try:
            gc.collect()
            if DEV == "cuda":
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"    (dọn dẹp lỗi, bỏ qua: {type(e).__name__})")
    if not cuda_con_song():
        print("\n!! CUDA CONTEXT ĐÃ HỎNG (device-side assert). Mọi model sau đều chết theo —")
        print("   lỗi loại này không cứu được trong cùng tiến trình. DỪNG SẠCH tại đây.")
        print(f"   {len(RES)} model đã chấm vẫn nằm nguyên trong {OUT}/.")
        print("   Tải về, upload lên dataset, chạy lại -> nó BỎ QUA những cái đã xong.")
        break
print(f"\nxong {len(RES)}/{len(MODELS)} model — TẢI {OUT}/ VỀ TRƯỚC KHI ĐÓNG PHIÊN")



In [ ]:
# ===== Bước 4: in bảng Markdown, dán thẳng vào báo cáo =====
rows = []
for tag, (per, mid, mt) in RES.items():
    a, b = do(per, 0), do(per, 1)
    rows.append((a["R@5"], tag, mid, mt.get("params"), a, b, mt.get("sec_per_pair")))
rows.sort(reverse=True)                      # xếp theo n=0: bộ chấm THUẦN, không pha rổ

f2 = lambda v, f: (f % v) if v is not None else "—"
print("| Model | Tham số | R@1 | **R@5 (thuần)** | R@10 | MRR@10 | R@5 (+RRF) | s/cặp |")
print("|---|---|---|---|---|---|---|---|")
for r5, tag, mid, pa, a, b, sp in rows:
    star = " ← đang dùng" if tag == "AITeamVN" else ""
    print(f"| `{mid}`{star} | {f2(pa and pa/1e9, '%.3fB')} | {a['R@1']:.4f} | "
          f"**{a['R@5']:.4f}** | {a['R@10']:.4f} | {a['MRR@10']:.4f} | "
          f"{b['R@5']:.4f} | {f2(sp, '%.3f')} |")

print("\n> Giao thức: dev300 · rổ fusion top-50 của D · 1 đoạn/văn bản (BM25 mức đoạn,")
print("> gộp 1800 ký tự) · chấm MỘT tầng, không đọc sâu · 15.000 cặp mỗi model · n_bm25=1.")
print("> Con số THẤP HƠN hệ thống hoàn chỉnh (0.9350) vì bảng này bỏ tầng đọc sâu —")
print("> nó dùng để xếp hạng bộ chấm với nhau, không phải để báo cáo điểm hệ thống.")
if "AITeamVN" in RES:
    top = rows[0]
    print(f"\nDẪN ĐẦU: {top[1]}  R@5 = {top[0]:.4f}")
    if top[1] != "AITeamVN":
        me = next(r for r in rows if r[1] == "AITeamVN")
        print(f"!! AITeamVN chỉ {me[0]:.4f} — KÉM {(top[0]-me[0])*100:.2f} điểm. "
              f"Nếu chênh ≥2,0 thì đây là cần gạt thật, không chỉ là dòng cho báo cáo.")